# 01 · Data prep

**Run this first.** It turns the 30 raw recordings into one clean file that every other notebook loads,
so the whole team works on identical data. (Notebooks 02–06 run this automatically if the file is missing.)

**What it does**
1. Loads each recording: 16 EMG channels at 2,000 samples/s and 20 joint angles
2. Turns raw EMG into a smooth **envelope** (how active each muscle area is) at 50 samples/s
3. Marks frames where pose tracking failed
4. Splits each recording in time: first 80% train, 1 s gap, last 20% test
5. Saves everything to `data/processed_mini.npz`

The code comes from `notebooks/eda/03_THANOS_DimRed.ipynb`, where it was reviewed.

## 🎯 Goal and milestones
*Draft milestones: agree them as a team on Friday, then edit here.*

**Goal:** give the whole team one identical, clean dataset, so every result in 02 to 06 is comparable.

**Why:** if each person filters or splits the data their own way, our numbers can't be compared and the final
summary falls apart. This notebook is the foundation the research question rests on.

**Milestones**
1. Runs on Colab and saves `data/processed_mini.npz` (30 recordings, 15 stages, 2 hands)
2. You can explain the envelope (filter → rectify → smooth) to the team in 1 minute
3. Printed checks look sane: % of frames with valid pose, train/test sizes

**Done when:** a teammate's notebook loads the file and prints the same numbers as yours.

In [ ]:
# ── Setup: works on Colab and locally ─────────────────────────────────────────
# After this cell the working directory is the repo root, wherever you run from.
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ or Path("/content").exists()
if IN_COLAB:
    ROOT = Path("/content/emg2pose")
    if not ROOT.exists():
        !git clone https://github.com/Wilbertbh-Tan/emg2pose.git {ROOT}
    !pip install -q h5py
else:
    ROOT = Path.cwd().resolve()
    while not (ROOT / "setup.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    assert (ROOT / "setup.py").exists(), "Run this notebook from inside the emg2pose repo"
os.chdir(ROOT)

PROCESSED = ROOT / "data" / "processed_mini.npz"   # made by notebook 01 (data/ is never committed)

DATA_DIR = ROOT / "emg2pose_dataset_mini"
if not DATA_DIR.exists():   # download the mini dataset (~617 MB) once
    url = "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_dataset_mini.tar"
    subprocess.run(["curl", "-s", url, "-o", "emg2pose_dataset_mini.tar"], check=True)
    subprocess.run(["tar", "-xf", "emg2pose_dataset_mini.tar"], check=True)
    os.remove("emg2pose_dataset_mini.tar")
print(f"{'Colab' if IN_COLAB else 'Local'} | repo root: {ROOT}")

In [ ]:
import glob
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

FS_OUT = 50          # samples per second after downsampling
TRAIN_FRACTION = 0.8
GAP = FS_OUT         # 1 s gap between train and test, so smoothing can't leak across

## 1. Load a recording
Each HDF5 file is one hand doing one task ("stage") for about a minute. The task and hand labels are
stored inside the file.

In [ ]:
def load_recording(path):
    with h5py.File(path, "r") as f:
        g = f["emg2pose"]
        ts = g["timeseries"]
        emg_raw = ts["emg"][:].astype(np.float32)                   # (T, 16)
        pose_deg = np.rad2deg(ts["joint_angles"][:].astype(np.float32))  # (T, 20)
        fs = 1.0 / np.median(np.diff(ts["time"][:1000]))
        return emg_raw, pose_deg, fs, str(g.attrs["stage"]), str(g.attrs["side"])

files = sorted(glob.glob(str(DATA_DIR / "*.hdf5")))
emg_raw, pose_deg, fs, stage, side = load_recording(files[0])
print(f"{len(files)} recordings. First: {stage}, {side} hand, {emg_raw.shape[0]/fs:.0f} s at {fs:.0f} Hz")

## 2. From raw EMG to an envelope
Raw EMG is a fast, noisy signal that swings positive and negative. Muscle *activity* is its size, so:

1. **Band-pass 20–450 Hz:** keep the frequencies where muscle signal lives (removes drift and high noise)
2. **Notch at 60 Hz:** remove mains electricity hum
3. **Rectify:** take the absolute value, so size is all that's left
4. **Low-pass at 10 Hz:** smooth it into an envelope
5. **Downsample to 50 Hz** (the smoothing removed everything faster, so nothing is lost)
6. **Scale per recording** so its 95th percentile is 1, because electrode contact differs between recordings

Zero-phase filtering (`filtfilt`) is used so the envelope isn't shifted in time (Lecture 2, p. 17).
The envelope is non-negative, which NMF needs.

In [ ]:
def to_envelope(emg_raw, fs, fs_out=FS_OUT):
    nyq = fs / 2
    sos_bp = signal.butter(4, [20 / nyq, min(450, 0.95 * nyq) / nyq], btype="bandpass", output="sos")
    b_notch, a_notch = signal.iirnotch(60.0, Q=30.0, fs=fs)
    sos_lp = signal.butter(4, 10 / nyq, btype="lowpass", output="sos")

    x = signal.sosfiltfilt(sos_bp, emg_raw, axis=0)
    x = signal.filtfilt(b_notch, a_notch, x, axis=0)
    env = signal.sosfiltfilt(sos_lp, np.abs(x), axis=0)
    env = np.clip(env, 1e-8, None)                 # strictly non-negative
    env = env[:: int(round(fs / fs_out))]           # downsample
    return (env / (np.percentile(env, 95) + 1e-8)).astype(np.float32)

env = to_envelope(emg_raw, fs)

# Look at it: 2 seconds of one channel, raw vs envelope
t_raw = np.arange(int(2 * fs)) / fs
t_env = np.arange(2 * FS_OUT) / FS_OUT
fig, ax = plt.subplots(2, 1, figsize=(10, 4), sharex=True)
ax[0].plot(t_raw, emg_raw[: len(t_raw), 0], lw=0.4); ax[0].set_ylabel("raw EMG ch0")
ax[1].plot(t_env, env[: len(t_env), 0], color="C1"); ax[1].set_ylabel("envelope ch0"); ax[1].set_xlabel("time (s)")
plt.tight_layout(); plt.show()

## 3. Bad pose frames and the train/test split
- When motion capture loses the hand, **all 20 angles are stored as exactly 0** (not NaN). Those frames are
  marked `pose_valid = False`. EMG-only analyses can still use them; anything with pose must skip them.
- Each recording is split **in time**: the first 80% is train, then a 1 s gap, then the rest is test.
  This tests whether results hold on *new moments of the same tasks*.

In [ ]:
cols = {k: [] for k in ["emg", "pose", "pose_valid", "rec", "split"]}
stages, sides, names = [], [], []

for r, path in enumerate(files):
    emg_raw, pose_deg, fs, stage, side = load_recording(path)
    env = to_envelope(emg_raw, fs)
    p = pose_deg[:: int(round(fs / FS_OUT))][: len(env)]
    env = env[: len(p)]

    valid = ~np.all(np.isclose(p, 0.0), axis=1) & ~np.isnan(p).any(axis=1)
    lab = np.zeros(len(env), dtype=np.int8)                 # 0 = train
    cut = int(TRAIN_FRACTION * len(env))
    lab[cut: cut + GAP] = -1                                # gap
    lab[cut + GAP:] = 1                                     # test

    for k, v in zip(cols, [env, p.astype(np.float32), valid, np.full(len(env), r, np.int16), lab]):
        cols[k].append(v)
    stages.append(stage); sides.append(side); names.append(Path(path).stem)

data = {k: np.concatenate(v) for k, v in cols.items()}
data.update(stage=np.array(stages), side=np.array(sides), filename=np.array(names), fs=np.array(FS_OUT))

## 4. Save and check

In [ ]:
PROCESSED.parent.mkdir(exist_ok=True)
np.savez_compressed(PROCESSED, **data)

d = np.load(PROCESSED)
assert d["emg"].shape[1] == 16 and d["pose"].shape[1] == 20
assert len({len(d[k]) for k in ["emg", "pose", "pose_valid", "rec", "split"]}) == 1
assert (d["emg"] >= 0).all(), "envelope must be non-negative for NMF"

print(f"Saved {PROCESSED.relative_to(ROOT)}: {len(d['emg'])} frames from {len(d['stage'])} recordings")
print(f"train {np.mean(d['split'] == 0):.0%} · test {np.mean(d['split'] == 1):.0%} · "
      f"pose valid {d['pose_valid'].mean():.1%}")
print(f"{len(set(d['stage']))} stages · hands: {sorted(set(map(str, d['side'])))}")